# 121 — Presupuestos de pasos, tokens, costo y tiempo

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Un agente tiene costo ABIERTO: cada iteración re-paga el contexto acumulado. Se
presupuestan **cuatro monedas por separado** — pasos, tokens, dinero, tiempo — y la
primera que se agote detiene la tarea (no son intercambiables).

**Modelo de costo** (bucle de n pasos, contexto inicial c0, Δ tokens nuevos por paso,
s tokens de salida por paso):

```text
entrada ≈ n·c0 + Δ·n·(n-1)/2     ← CUADRÁTICO en n si no se compacta
salida  ≈ n·s                     ← lineal
costo   = entrada·p_in + salida·p_out   (p_out suele ser varias veces p_in)
```

Consecuencia: duplicar los pasos casi cuadruplica los tokens de entrada. Presupuesto
de tokens y gestión de contexto (118) son la misma batalla.

### 📉 Contrato de tres fases

1. **Estimar (antes):** presupuesto por sub-tarea del plan (115) + reserva (~20 %).
2. **Medir (durante):** telemetría por paso — spans con tokens, costo y estado;
   alertas al 80 % de cualquier moneda, ANTES del corte.
3. **Actuar (al agotarse):** el presupuesto se comprueba ANTES de cada acción; parada
   limpia con checkpoint (118) + estado parcial + reporte de qué falta.

Los reintentos (116) consumen del mismo pozo y son señal diagnóstica, no ruido. El
laboratorio `observability` emite la fase "medir" mínima: 3 spans con tokens
(120 + 80 + 40 = 240) y duración — los datos sobre los que se corta o alerta.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — Lee la telemetría.** Ejecuta `run_lab("observability", seed=121)` y
verifica con código que `total_tokens` es la suma de los spans. ¿Qué span consumió más
y qué fracción del total representa? ¿Qué le falta a cada span para poder facturarlo?

**Ejercicio 2 — Presupuesto a mano.** Un agente de investigación hace n=8 pasos con
c0=3.000 tokens, Δ=1.200 por paso y s=500 de salida; precios: 3 USD/M entrada,
15 USD/M salida. Calcula A MANO (sin código primero): tokens de entrada, de salida y
costo total. Después añade reserva del 20 % y expresa el presupuesto en las cuatro
monedas (asume 5 s por llamada + 20 s de tools para el tiempo).

**Ejercicio 3 — Sensibilidad al doble de pasos.** Con los datos del Ejercicio 2,
recalcula para n=16. ¿Por qué el costo NO se duplica? ¿Qué Δ efectivo tendría que
lograr la compactación (118) para que n=16 costara como el n=8 original (aprox.)?

**Ejercicio 4 — Implementa el guardián.** Programa `Guardian(pasos_max, tokens_max)`
con método `puede_ejecutar(tokens_estimados)` que: permita la acción solo si TODAS las
monedas alcanzan, alerte al cruzar el 80 % de cualquiera y, al denegar, produzca el
reporte de parada limpia (moneda agotada, consumido, restante). Demuéstralo con los
spans del laboratorio más un cuarto paso de 100 tokens con presupuesto de 300.

In [ ]:
# TODO: ejecuta run_lab("observability", seed=121)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicios 1 y 2
result = run_lab("observability", seed=121)
spans = result["result"]["spans"]
# 1) verifica total_tokens == suma de spans; span dominante y su fraccion
# 2) presupuesto a mano (escribe primero los numeros en papel):
n, c0, delta, s = 8, 3000, 1200, 500
P_IN, P_OUT = 3 / 1_000_000, 15 / 1_000_000
tokens_entrada = None   # n*c0 + delta*n*(n-1)/2
tokens_salida = None    # n*s
costo = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 4: guardián de presupuesto
class Guardian:
    def __init__(self, pasos_max, tokens_max):
        self.pasos_max, self.tokens_max = pasos_max, tokens_max
        self.pasos, self.tokens = 0, 0

    def puede_ejecutar(self, tokens_estimados):
        # comprueba ANTES de actuar; alerta al 80 %; reporte al denegar
        pass

    def registrar(self, tokens_reales):
        pass


## Reflexión

1. ¿Por qué "comprobar el presupuesto después de actuar" es un error de diseño y qué
   relación tiene con los puntos consistentes de checkpoint de la clase 118?
2. El laboratorio muestra tokens por span pero no costo en dinero. ¿Qué dos datos
   externos necesitas para convertir spans en factura, y por qué deben versionarse?
3. Tu agente se detuvo por presupuesto al 100 % de tokens con 40 % de los pasos
   usados. ¿Qué diagnóstico sugiere esa asimetría y qué arreglo de la clase 118
   probarías primero?